# Transaction amount outlier exploration

Standalone extraction of the transaction-quality and outlier work from `imputation.ipynb`.

Run top to bottom. This notebook compares raw and log-transformed transaction amounts and explores the original **$8,000 upper cutoff**. This is a manual exploratory threshold, not a validated fraud rule. No source data is overwritten and the threshold is not applied to fraud modelling.

Input: `project2/data/tables/processed/transactions_clean`.


## 1. Setup


In [ ]:

# Resolve imports by repository contents, regardless of notebook location.
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scripts" / "process_cleaning.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project repository")
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F

# Works when launched from the repository, project2, or notebooks directory.

PROCESSED_PATH = PROJECT_ROOT / "data" / "tables" / "processed"
if not PROCESSED_PATH.is_dir():
    raise FileNotFoundError("Set PROJECT_ROOT to your project2 directory.")
RANDOM_STATE = 42
spark = SparkSession.builder.config("spark.sql.session.timeZone", "Etc/UTC").config("spark.driver.memory", "6g").getOrCreate()
from scipy.stats import norm
AMOUNT_CUTOFF = 8000.0
MAX_PLOT_ROWS = 100_000


## 2. Duplicate audit

Preserve the original duplicate key. It may merge distinct purchases with equal amounts for the same user and timestamp; review identifiers before adopting it permanently.


In [ ]:
trans_raw = spark.read.parquet(str(PROCESSED_PATH / "transactions_clean"))
DUPLICATE_KEY = ["order_datetime", "user_id", "dollar_value"]
trans = trans_raw.dropDuplicates(DUPLICATE_KEY)
raw_n, deduplicated_n = trans_raw.count(), trans.count()
print({"input_rows": raw_n, "deduplicated_rows": deduplicated_n,
       "duplicate_rows_removed": raw_n - deduplicated_n})


## 3. Amount validity and raw distribution

Separate missing, nonfinite, and negative amounts before `log1p`. Negative values may be refunds and should be investigated separately.


In [ ]:
value = F.col("dollar_value").cast("double")
valid = value.isNotNull() & ~F.isnan(value) & (F.abs(value) != float("inf")) & (value >= 0)
cost = trans.select(value.alias("dollar_value")).filter(valid)
valid_n = cost.count()
print({"valid_nonnegative_amounts": valid_n,
       "missing_nonfinite_or_negative": deduplicated_n - valid_n})
if valid_n == 0:
    raise ValueError("No valid nonnegative amounts to analyse.")
cost.summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

# Bound driver-memory use; plots use a seeded Spark sample, capped at MAX_PLOT_ROWS.
fraction = min(1.0, MAX_PLOT_ROWS / valid_n)
plot_values = cost.sample(False, fraction, seed=RANDOM_STATE).limit(MAX_PLOT_ROWS).toPandas()["dollar_value"].to_numpy()
if len(plot_values) == 0:
    raise ValueError("Plot sample is empty; increase MAX_PLOT_ROWS.")
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(plot_values, bins=30)
ax.set(xlabel="Transaction amount ($)", ylabel="Sample count", title="Raw amount distribution")
plt.show()


## 4. Log-scale distribution


In [ ]:
log_cost = cost.withColumn("log_cost", F.log1p("dollar_value"))
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.log1p(plot_values), bins=30)
ticks = np.arange(0, int(np.ceil(np.log1p(plot_values.max()))) + 1)
ax.set_xticks(ticks, [f"{np.expm1(t):,.0f}" for t in ticks], rotation=45)
ax.set(xlabel="Transaction amount ($; log1p spacing)", ylabel="Sample count",
       title="Log-transformed amount distribution")
fig.tight_layout()
plt.show()


## 5. Explore the manual upper cutoff

Report counts against the valid-amount population. Invalid amounts were handled separately above. A large purchase is not necessarily an error or fraud.


In [ ]:
clean_cost = log_cost.filter(F.col("dollar_value") <= AMOUNT_CUTOFF)
high_cost = log_cost.filter(F.col("dollar_value") > AMOUNT_CUTOFF)
kept_n = clean_cost.count()
flagged_n = valid_n - kept_n
print({"cutoff": AMOUNT_CUTOFF, "retained": kept_n, "above_cutoff": flagged_n,
       "above_cutoff_pct_of_valid": 100 * flagged_n / valid_n})
log_stats = clean_cost.agg(F.mean("log_cost").alias("mean"),
                           F.variance("log_cost").alias("variance")).first()


## 6. Retained log amounts versus a fitted normal curve

The curve uses the full retained population's mean and variance; the histogram uses the bounded sample. Visual resemblance to a normal distribution does not validate the cutoff.


In [ ]:
retained_values = np.log1p(plot_values[plot_values <= AMOUNT_CUTOFF])
mu, variance = log_stats["mean"], log_stats["variance"]
if len(retained_values) < 2 or variance is None or variance <= 0:
    print("Not enough variation below the cutoff for a fitted normal comparison.")
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(retained_values, bins=30, density=True, alpha=0.6, color="gray", label="Retained sample")
    x = np.linspace(retained_values.min(), retained_values.max(), 200)
    ax.plot(x, norm.pdf(x, mu, np.sqrt(variance)), color="red", label="Fitted normal")
    ax.set(xlabel="log1p(transaction amount)", ylabel="Density", title="Retained log amounts")
    ax.legend()
    plt.show()


## Conclusions to record after running

Document the duplicate and invalid-amount rates, the share above the cutoff, and why a chosen threshold is appropriate. Compare thresholds before deciding whether to flag, cap, retain, or exclude large amounts. Nothing is automatically removed from the stored transactions or the fraud-modelling notebook.
